In [2]:
import pandas as pd

# 데이터 로드
orders = pd.read_csv('../data/olist_orders_dataset.csv')
items = pd.read_csv('../data/olist_order_items_dataset.csv')
reviews = pd.read_csv('../data/olist_order_reviews_dataset.csv')
customers = pd.read_csv('../data/olist_customers_dataset.csv')
products = pd.read_csv('../data/olist_products_dataset.csv')
sellers = pd.read_csv('../data/olist_sellers_dataset.csv')
payments = pd.read_csv('../data/olist_order_payments_dataset.csv')
geo = pd.read_csv('../data/olist_geolocation_dataset.csv', encoding='latin-1')

# 기본 구조 확인
for name, df in [('orders', orders), ('items', items), ('reviews', reviews),
                  ('customers', customers), ('products', products),
                  ('sellers', sellers), ('payments', payments), ('geo', geo)]:
    print(f"\n=== {name} ===")
    print(f"shape: {df.shape}")
    print(f"columns: {list(df.columns)}")
    print(f"null 수:\n{df.isnull().sum()}")


=== orders ===
shape: (99441, 8)
columns: ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']
null 수:
order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

=== items ===
shape: (112650, 7)
columns: ['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']
null 수:
order_id               0
order_item_id          0
product_id             0
seller_id              0
shipping_limit_date    0
price                  0
freight_value          0
dtype: int64

=== reviews ===
shape: (99224, 7)
columns: ['review_id', 'order_id', 'review_score', 'review_co

In [5]:
timestamp_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col in timestamp_cols:
    orders[col] = pd.to_datetime(orders[col])

items['shipping_limit_date'] = pd.to_datetime(items['shipping_limit_date'])

orders['payment_approval_time'] = (orders['order_approved_at'] - orders['order_purchase_timestamp']).dt.total_seconds() / 3600
orders['seller_processing_time'] = (orders['order_delivered_carrier_date'] - orders['order_approved_at']).dt.total_seconds() / 3600
orders['carrier_delivery_time'] = (orders['order_delivered_customer_date'] - orders['order_delivered_carrier_date']).dt.total_seconds() / 3600
orders['total_delivery_time'] = (orders['order_delivered_customer_date'] - orders['order_purchase_timestamp']).dt.total_seconds() / 3600
orders['delivery_delay'] = (orders['order_delivered_customer_date'] - orders['order_estimated_delivery_date']).dt.total_seconds() / 3600


items_orders =pd.merge(items, orders , on='order_id', how='inner')
items_orders['seller_deadline_diff'] = (items_orders['order_delivered_carrier_date'] - items_orders['shipping_limit_date']).dt.total_seconds() / 3600

incomplete_rate = orders['order_delivered_customer_date'].isnull().sum() / len(orders)
print(incomplete_rate)

#items_orders['seller_deadline_diff'].head()
#orders.head()
#len(orders[orders['delivery_delay'] > 0])

0.02981667521444877


In [11]:
orders['order_status'].value_counts()
orders_delivered = orders[orders['order_status'] == 'delivered']

orders_delivered.isnull().sum()
orders_delivered = orders_delivered.dropna(subset=['order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date'])
orders_delivered.shape

orders_delivered.isnull().sum() 


order_id                         0
customer_id                      0
order_status                     0
order_purchase_timestamp         0
order_approved_at                0
order_delivered_carrier_date     0
order_delivered_customer_date    0
order_estimated_delivery_date    0
payment_approval_time            0
seller_processing_time           0
carrier_delivery_time            0
total_delivery_time              0
delivery_delay                   0
dtype: int64